# 双向循环神经网络（Bidirectional RNN）

## 一、问题背景

在标准的序列建模中（如普通RNN、GRU、LSTM），我们通常假设在给定观测的情况下 （例如，在时间序列的上下文中或在语言模型的上下文中）， 对下一个输出进行建模。
即当前时刻的预测仅依赖“过去”。

但在很多实际任务中，这种假设是不充分的。例如：

- 填空问题：“我 ___ 饿了”
- 命名实体识别：判断 “Green” 是名字还是颜色
- 语音/文本理解：一个词的含义往往依赖后文

这些任务本质上需要同时利用过去和未来的信息。

## 二、从概率模型到双向建模

在隐马尔可夫模型（HMM）中，我们有：

$$P(x_1, \ldots, x_T, h_1, \ldots, h_T) = \prod_{t=1}^T P(h_t \mid h_{t-1}) P(x_t \mid h_t), \text{ where } P(h_1 \mid h_0) = P(h_1).$$

为了计算边缘概率，可以通过两种递归：

前向递归（Forward）
$$\pi_{t+1}(h_{t+1}) = \sum_{h_t} \pi_t(h_t) P(x_t \mid h_t) P(h_{t+1} \mid h_t).$$

递归被初始化为$\pi_1(h_1) = P(h_1)$。
符号简化，也可以写成$\pi_{t+1} = f(\pi_t, x_t)$，
其中$f$是一些可学习的函数。

后向递归（Backward）

$$\rho_{t-1}(h_{t-1})= \sum_{h_{t}} P(h_{t} \mid h_{t-1}) P(x_{t} \mid h_{t}) \rho_{t}(h_{t}),$$

初始化$\rho_T(h_T) = 1$。
前向和后向递归都允许我们对$T$个隐变量在$\mathcal{O}(kT)$
（线性而不是指数）时间内对$(h_1, \ldots, h_T)$的所有值求和。

结合前向和后向递归，最终可以得到：
$$P(x_j \mid x_{-j}) \propto \sum_{h_j} \pi_j(h_j) \rho_j(h_j) P(x_j \mid h_j).$$


前向信息来自过去，后向信息来自未来，双向RNN就是这个思想的“神经网络版本”。

## 三、双向RNN的基本结构

双向RNN由两个独立的RNN组成：

1. 前向RNN（forward RNN）
2. 反向RNN（backward RNN）

与隐马尔可夫模型中的动态规划的前向和后向递归没有太大区别。 其主要区别是，在隐马尔可夫模型中的方程具有特定的统计意义。 双向循环神经网络没有这样容易理解的解释， 我们只能把它们当作通用的、可学习的函数。

## 四、数学定义

对于任意时间步$t$，给定一个小批量的输入数据
$\mathbf{X}_t \in \mathbb{R}^{n \times d}$
（样本数$n$，每个示例中的输入数$d$），
并且令隐藏层激活函数为$\phi$

在双向架构中，我们设该时间步的前向和反向隐状态分别为
$\overrightarrow{\mathbf{H}}_t  \in \mathbb{R}^{n \times h}$和
$\overleftarrow{\mathbf{H}}_t  \in \mathbb{R}^{n \times h}$，
其中$h$是隐藏单元的数目。
前向和反向隐状态的更新如下：
$$
\begin{aligned}
\overrightarrow{\mathbf{H}}_t &= \phi(\mathbf{X}_t \mathbf{W}_{xh}^{(f)} + \overrightarrow{\mathbf{H}}_{t-1} \mathbf{W}_{hh}^{(f)}  + \mathbf{b}_h^{(f)}),\\
\overleftarrow{\mathbf{H}}_t &= \phi(\mathbf{X}_t \mathbf{W}_{xh}^{(b)} + \overleftarrow{\mathbf{H}}_{t+1} \mathbf{W}_{hh}^{(b)}  + \mathbf{b}_h^{(b)}),
\end{aligned}
$$

其中，权重$\mathbf{W}_{xh}^{(f)} \in \mathbb{R}^{d \times h}, \mathbf{W}_{hh}^{(f)} \in \mathbb{R}^{h \times h}, \mathbf{W}_{xh}^{(b)} \in \mathbb{R}^{d \times h}, \mathbf{W}_{hh}^{(b)} \in \mathbb{R}^{h \times h}$
和偏置$\mathbf{b}_h^{(f)} \in \mathbb{R}^{1 \times h}, \mathbf{b}_h^{(b)} \in \mathbb{R}^{1 \times h}$都是模型参数。

前向隐状态是从左到右传播（从过去到现在）

反向隐状态是从右到左传播（从未来到现在）

拼接隐藏状态
最后，输出层计算得到的输出为
$\mathbf{O}_t \in \mathbb{R}^{n \times q}$（$q$是输出单元的数目）：

$$\mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{hq} + \mathbf{b}_q.$$

这里，权重矩阵$\mathbf{W}_{hq} \in \mathbb{R}^{2h \times q}$
和偏置$\mathbf{b}_q \in \mathbb{R}^{1 \times q}$
是输出层的模型参数。

## 五、信息流动方式

在双向RNN中，信息有两种传播路径：

1. 时间方向包括正向和反向

2. 特征融合，两个方向的隐藏状态在当前时间步融合


## 六、优势

能利用完整上下文（context），对语义理解更准确，对不确定词（歧义词）判断更好


## 七、缺点

无法用于“预测未来”：
双向循环神经网络的一个关键特性是：使用来自序列两端的信息来估计输出。 也就是说，我们使用来自过去和未来的观测信息来预测当前的观测。
但是在对下一个词元进行预测的情况中，这样的模型并不是我们所需的。
在训练期间，我们能够利用过去和未来的数据来估计现在空缺的词； 而在测试期间，我们只有过去的数据，因此精度将会很差。

训练/推理成本高：
其主要原因是网络的前向传播需要在双向层中进行前向和后向递归， 并且网络的反向传播还依赖于前向传播的结果。

实时任务不适用双向层的使用在实践中非常少，并且仅仅应用于部分场合。 例如，填充缺失的单词、词元注释（例如，用于命名实体识别） 以及作为序列处理流水线中的一个步骤对序列进行编码（例如，用于机器翻译）。


In [ ]:
import math
import time
import re
import requests
from collections import Counter

import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


# =========================
# 1. 读取 time machine 数据
# =========================
url = "http://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt"
file_path = "timemachine.txt"

response = requests.get(url)
with open(file_path, "wb") as f:
    f.write(response.content)

def read_time_machine():
    with open(file_path, "r") as f:
        lines = f.readlines()
    return [re.sub("[^A-Za-z]+", " ", line).strip().lower() for line in lines]

lines = read_time_machine()

# 字符级分词
tokens = [list(line) for line in lines]
corpus = [token for line in tokens for token in line]


# =========================
# 2. 构建词表
# =========================
class Vocab:
    def __init__(self, tokens):
        counter = Counter(tokens)
        self.idx_to_token = list(counter.keys())
        self.token_to_idx = {token: i for i, token in enumerate(self.idx_to_token)}

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, token):
        return self.token_to_idx[token]

    def to_tokens(self, indices):
        if isinstance(indices, (list, tuple)):
            return [self.idx_to_token[int(i)] for i in indices]
        return self.idx_to_token[int(indices)]

vocab = Vocab(corpus)
corpus = [vocab[token] for token in corpus]


# =========================
# 3. 构造数据迭代器
# =========================
def load_data_time_machine(batch_size, num_steps):
    corpus_tensor = torch.tensor(corpus, dtype=torch.long)

    num_tokens = (len(corpus_tensor) // batch_size) * batch_size
    corpus_tensor = corpus_tensor[:num_tokens]
    corpus_tensor = corpus_tensor.reshape(batch_size, -1)

    num_batches = (corpus_tensor.shape[1] - 1) // num_steps

    data = []
    for i in range(0, num_batches * num_steps, num_steps):
        X = corpus_tensor[:, i:i + num_steps]
        Y = corpus_tensor[:, i + 1:i + num_steps + 1]
        data.append((X, Y))
    return data, vocab


# =========================
# 4. 双向/单向 RNN 通用封装
# =========================
class RNNModel(nn.Module):
    def __init__(self, rnn_layer, vocab_size):
        super().__init__()
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = rnn_layer.hidden_size
        self.num_directions = 2 if rnn_layer.bidirectional else 1
        self.linear = nn.Linear(self.num_hiddens * self.num_directions, vocab_size)

    def forward(self, inputs, state):
        # inputs: (batch_size, num_steps)
        X = F.one_hot(inputs.T.long(), self.vocab_size).to(torch.float32)
        # X: (num_steps, batch_size, vocab_size)
        Y, state = self.rnn(X, state)
        # Y: (num_steps, batch_size, num_hiddens * num_directions)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, batch_size=1, device="cpu"):
        if isinstance(self.rnn, nn.LSTM):
            return (
                torch.zeros(
                    (self.num_directions * self.rnn.num_layers, batch_size, self.num_hiddens),
                    device=device
                ),
                torch.zeros(
                    (self.num_directions * self.rnn.num_layers, batch_size, self.num_hiddens),
                    device=device
                )
            )
        else:
            return torch.zeros(
                (self.num_directions * self.rnn.num_layers, batch_size, self.num_hiddens),
                device=device
            )


# =========================
# 5. 梯度裁剪
# =========================
def grad_clipping(net, theta):
    params = [p for p in net.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params if p.grad is not None))
    if norm > theta:
        for param in params:
            if param.grad is not None:
                param.grad[:] *= theta / norm


# =========================
# 6. 文本生成函数
# =========================
def predict_ch8(prefix, num_preds, net, vocab, device):
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]

    get_input = lambda: torch.tensor([[outputs[-1]]], device=device)

    # 预热
    for y in prefix[1:]:
        _, state = net(get_input(), state)
        outputs.append(vocab[y])

    # 预测
    for _ in range(num_preds):
        y, state = net(get_input(), state)
        pred = int(y.argmax(dim=1).reshape(1))
        outputs.append(pred)

    return ''.join(vocab.to_tokens(outputs))


# =========================
# 7. 训练一个 epoch
# =========================
def train_epoch_ch8(net, train_iter, loss, updater, device):
    state = None
    metric = [0.0, 0]
    start = time.time()

    for X, Y in train_iter:
        if state is None:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(state, tuple):
                state = tuple(s.detach() for s in state)
            else:
                state = state.detach()

        X, Y = X.to(device), Y.to(device)
        y = Y.T.reshape(-1)

        y_hat, state = net(X, state)
        l = loss(y_hat, y.long()).mean()

        updater.zero_grad()
        l.backward()
        grad_clipping(net, 1)
        updater.step()

        metric[0] += l.item() * y.numel()
        metric[1] += y.numel()

    ppl = math.exp(metric[0] / metric[1])
    speed = metric[1] / (time.time() - start)
    return ppl, speed


# =========================
# 8. 总训练函数
# =========================
def train_ch8(net, train_iter, vocab, lr, num_epochs, device):
    loss = nn.CrossEntropyLoss()
    updater = torch.optim.SGD(net.parameters(), lr=lr)

    ppl_list = []
    epoch_list = []

    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(net, train_iter, loss, updater, device)

        if (epoch + 1) % 10 == 0:
            print(f"epoch {epoch+1}, perplexity {ppl:.3f}")
            print(predict_ch8("time traveller", 50, net, vocab, device))
            ppl_list.append(ppl)
            epoch_list.append(epoch + 1)

    print(f"\n最终困惑度 {ppl:.1f}, {speed:.1f} 词元/秒, device={device}")
    print(predict_ch8("time traveller", 50, net, vocab, device))
    print(predict_ch8("traveller", 50, net, vocab, device))

    plt.figure(figsize=(6, 4))
    plt.plot(epoch_list, ppl_list)
    plt.xlabel("epoch")
    plt.ylabel("perplexity")
    plt.grid()
    plt.show()


# =========================
# 9. 加载数据
# =========================
batch_size, num_steps = 32, 35
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_iter, vocab = load_data_time_machine(batch_size, num_steps)


# =========================
# 10. 定义双向 LSTM
# =========================
vocab_size, num_hiddens, num_layers = len(vocab), 256, 2
num_inputs = vocab_size

lstm_layer = nn.LSTM(
    input_size=num_inputs,
    hidden_size=num_hiddens,
    num_layers=num_layers,
    bidirectional=True
)

model = RNNModel(lstm_layer, vocab_size).to(device)


# =========================
# 11. 训练模型
# =========================
num_epochs, lr = 500, 1
train_ch8(model, train_iter, vocab, lr, num_epochs, device)

epoch 10, perplexity 1.205
time travellerererererererererererererererererererererererererer
epoch 20, perplexity 1.131
time travellerererererererererererererererererererererererererer
epoch 30, perplexity 1.097
time travellerererererererererererererererererererererererererer
